# Intra-Chip vs Inter-Chip Curve Similarity / Separability

Companion to `20260816-lofo_data_coverage_diagnosis.ipynb` (which argues the "is it
the data" case numerically and via a 2D t-SNE projection) -- this one looks directly
at raw curve shapes, at four different groupings, per LOFO group:

1. **Intra-chip** (one figure per group, N subplots = N chips):
   - **1a same-target similarity**: each chip's own subplot, colored by *well*.
   - **1b different-target separability**: each chip's own subplot, colored by
     *target* (shared color-per-label across all subplots, so the same target is
     always the same color everywhere in this notebook).
2. **Inter-chip** (one figure per group):
   - **2a same-target similarity**: N subplots = N targets, each colored by *chip*
     (shared color-per-chip).
   - **2b pooled separability**: one plot, raw cloud colored by *target* (answers
     "does target separate when everything's pooled"), with one high-alpha mean line
     **per chip** inside each target's color -- chip-driven drift shows up as lines
     splitting apart within one color group, without needing a second color channel.

Every raw-curve panel is **subsampled** (not literally every pixel -- a chip/well can
carry thousands) at low alpha; mean lines are always the full, unsampled mean.
PC excluded everywhere (control, not a target).

**Quantitative companion** (§5-6): a 4-level distance decomposition (intra-well
pixel noise -> inter-well same-chip-same-label -> inter-chip same-label ->
inter-label pooled -- generalizes `20260816`'s within/cross-chip distance ratio to a
scale that doesn't require rare same-(target,concentration) replicate wells), plus a
raw-curve-space (PCA-reduced, not forced to 2D) silhouette score by well/chip/label,
complementing the earlier t-SNE-projection silhouette with one computed without a 2D
projection's potential distortion.

In [ ]:
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

cdt = importlib.import_module("04_cross_dataset_training")

print("Imports OK.")

## 1. Configuration

In [ ]:
GROUPS_TO_TRY = list(config.CROSS_DATASET_GROUPS.keys())[3:11]
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
CURVE_TYPE = "ori_curve_norm"

N_SUBSAMPLE = 300     # max raw curves drawn per well/chip/label panel -- mean lines always use every curve
ALPHA_RAW = 0.06
ALPHA_MEAN = 0.95

SAVE_SUBDIR = "curve_similarity_plots"

def short_name(folder):
    return folder.split('_U_', 1)[1]

print(f"Groups ({len(GROUPS_TO_TRY)}): {GROUPS_TO_TRY}")

## 2. Helpers -- data loading & palettes

`load_group_pool` reuses `cdt.combine_group` (same call `20260816`'s §5 and the
UMAP/t-SNE section make) -- pools the group's chips onto one resampled time grid,
applying that group's own `LOFO_EXCLUDE_WELL_MAPPING`. `well_ids` come back as
`"{chip}::{well}"` (`04_cross_dataset_training.py:load_curve_data`); stripped down to
just the well part here since `dataset_id` already carries the chip. Palettes are
built once per group and reused across all four figures so a given target/chip is
always the same color everywhere in that group's output.

In [ ]:
def _sub_idx(idx_pool, n_max, rng):
    idx_pool = np.asarray(idx_pool)
    if len(idx_pool) <= n_max:
        return idx_pool
    return rng.choice(idx_pool, size=n_max, replace=False)


def load_group_pool(group_name, curve_type=CURVE_TYPE):
    folder_names = config.CROSS_DATASET_GROUPS[group_name]
    exp_paths_g = [Path(EXP_FOLDER, name) for name in folder_names]
    combined = cdt.combine_group(exp_paths_g, group_name, curve_type=curve_type)
    if combined is None:
        return None

    curves = np.asarray(combined["curves"])
    y_mapped = np.asarray(combined["Y_mapped"])
    dataset_id = np.asarray([short_name(str(d)) for d in combined["dataset_id"]])
    well_ids_raw = combined.get("well_ids")
    well_ids = (np.asarray([str(w).split("::", 1)[-1] for w in well_ids_raw])
               if well_ids_raw is not None else None)
    t_grid = combined["resampler"].t_grid

    mask = y_mapped != "PC"
    return {
        "curves": curves[mask], "y_mapped": y_mapped[mask], "dataset_id": dataset_id[mask],
        "well_ids": well_ids[mask] if well_ids is not None else None, "t_grid": t_grid,
    }


def build_label_palette(labels):
    labels = sorted(set(labels) - {"PC"})
    cmap = plt.cm.tab10 if len(labels) <= 10 else plt.cm.tab20
    return {l: cmap(i % cmap.N) for i, l in enumerate(labels)}


def build_chip_palette(chips):
    chips = sorted(set(chips))
    cmap = plt.cm.Set1
    return {c: cmap(i % 9) for i, c in enumerate(chips)}


print("Data/palette helpers defined.")

## 3. Visual -- intra-chip (1a, 1b)

In [ ]:
def plot_intra_chip_by_well(group_name, pool, save_path, n_sub=N_SUBSAMPLE, seed=0):
    curves, y_mapped, dataset_id, well_ids, t_grid = (
        pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["well_ids"], pool["t_grid"])
    if well_ids is None:
        print(f"  [SKIP 1a] {group_name}: no well_ids -- can't color by well.")
        return
    rng = np.random.default_rng(seed)
    chips = sorted(set(dataset_id))
    fig, axes = plt.subplots(1, len(chips), figsize=(5 * len(chips), 5), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, chip in zip(axes, chips):
        chip_mask = dataset_id == chip
        wells_here = sorted(set(well_ids[chip_mask]))
        cmap = plt.cm.tab20
        well_colors = {w: cmap(i % 20) for i, w in enumerate(wells_here)}
        for w in wells_here:
            idx = np.where(chip_mask & (well_ids == w))[0]
            idx_sub = _sub_idx(idx, n_sub, rng)
            label_here = y_mapped[idx][0] if len(idx) else "?"
            ax.plot(t_grid, curves[idx_sub].T, color=well_colors[w], alpha=ALPHA_RAW, linewidth=0.6)
            ax.plot(t_grid, curves[idx].mean(axis=0), color=well_colors[w], alpha=ALPHA_MEAN, linewidth=2.2,
                   label=f"well {w} ({label_here})")
        ax.set_title(chip, fontsize=10, fontweight="bold")
        ax.legend(fontsize=7, loc="best")

    axes[0].set_ylabel("Signal")
    fig.text(0.5, 0.02, "Time", ha="center")
    fig.suptitle(f"{group_name} -- intra-chip, colored by WELL (low alpha=pixels, high alpha=well mean)", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


def plot_intra_chip_by_label(group_name, pool, label_palette, save_path, n_sub=N_SUBSAMPLE, seed=0):
    curves, y_mapped, dataset_id, t_grid = pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["t_grid"]
    rng = np.random.default_rng(seed)
    chips = sorted(set(dataset_id))
    fig, axes = plt.subplots(1, len(chips), figsize=(5 * len(chips), 5), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, chip in zip(axes, chips):
        chip_mask = dataset_id == chip
        for lab in sorted(set(y_mapped[chip_mask])):
            idx = np.where(chip_mask & (y_mapped == lab))[0]
            idx_sub = _sub_idx(idx, n_sub, rng)
            ax.plot(t_grid, curves[idx_sub].T, color=label_palette[lab], alpha=ALPHA_RAW, linewidth=0.6)
            ax.plot(t_grid, curves[idx].mean(axis=0), color=label_palette[lab], alpha=ALPHA_MEAN, linewidth=2.2)
        ax.set_title(chip, fontsize=10, fontweight="bold")

    axes[0].set_ylabel("Signal")
    handles = [plt.Line2D([0], [0], color=c, lw=2.5, label=l) for l, c in label_palette.items()]
    fig.legend(handles=handles, loc="upper center", ncol=min(len(label_palette), 8), bbox_to_anchor=(0.5, 1.08), fontsize=9)
    fig.text(0.5, 0.02, "Time", ha="center")
    fig.suptitle(f"{group_name} -- intra-chip, colored by TARGET (low alpha=pixels, high alpha=target mean)", fontsize=12, y=1.14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


print("1a/1b plot functions defined.")

## 4. Visual -- inter-chip (2a, 2b)

In [ ]:
def plot_inter_chip_by_chip(group_name, pool, chip_palette, save_path, n_sub=N_SUBSAMPLE, seed=0):
    curves, y_mapped, dataset_id, t_grid = pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["t_grid"]
    rng = np.random.default_rng(seed)
    labels = sorted(set(y_mapped))
    fig, axes = plt.subplots(1, len(labels), figsize=(5 * len(labels), 5), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, lab in zip(axes, labels):
        lab_mask = y_mapped == lab
        for chip in sorted(set(dataset_id[lab_mask])):
            idx = np.where(lab_mask & (dataset_id == chip))[0]
            idx_sub = _sub_idx(idx, n_sub, rng)
            ax.plot(t_grid, curves[idx_sub].T, color=chip_palette[chip], alpha=ALPHA_RAW, linewidth=0.6)
            ax.plot(t_grid, curves[idx].mean(axis=0), color=chip_palette[chip], alpha=ALPHA_MEAN, linewidth=2.2)
        ax.set_title(lab, fontsize=10, fontweight="bold")

    axes[0].set_ylabel("Signal")
    handles = [plt.Line2D([0], [0], color=c, lw=2.5, label=ch) for ch, c in chip_palette.items()]
    fig.legend(handles=handles, loc="upper center", ncol=min(len(chip_palette), 6), bbox_to_anchor=(0.5, 1.1), fontsize=9)
    fig.text(0.5, 0.02, "Time", ha="center")
    fig.suptitle(f"{group_name} -- inter-chip, colored by CHIP (low alpha=pixels, high alpha=chip mean)", fontsize=12, y=1.16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.88])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


def plot_inter_chip_pooled(group_name, pool, label_palette, save_path, n_sub_per_label=N_SUBSAMPLE, seed=0):
    curves, y_mapped, dataset_id, t_grid = pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["t_grid"]
    rng = np.random.default_rng(seed)
    labels = sorted(set(y_mapped))
    fig, ax = plt.subplots(figsize=(9, 6))

    for lab in labels:
        idx = np.where(y_mapped == lab)[0]
        idx_sub = _sub_idx(idx, n_sub_per_label, rng)
        ax.plot(t_grid, curves[idx_sub].T, color=label_palette[lab], alpha=ALPHA_RAW, linewidth=0.5)
        for chip in sorted(set(dataset_id[idx])):
            idx_c = np.where((y_mapped == lab) & (dataset_id == chip))[0]
            ax.plot(t_grid, curves[idx_c].mean(axis=0), color=label_palette[lab], alpha=ALPHA_MEAN, linewidth=2.0)

    ax.set_xlabel("Time"); ax.set_ylabel("Signal")
    handles = [plt.Line2D([0], [0], color=c, lw=2.5, label=l) for l, c in label_palette.items()]
    ax.legend(handles=handles, loc="best", fontsize=9, title="Target (each color's multiple mean lines = different chips)")
    ax.set_title(f"{group_name} -- pooled, colored by TARGET\n(low alpha=all pixels; high alpha=one mean line per chip within each color)",
                fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


print("2a/2b plot functions defined.")

## 5. Quantitative -- 4-level distance decomposition

Generalizes `20260816`'s within/cross-chip distance ratio (which needed >=2 replicate
wells of the exact same (target, concentration) -- structurally ~0 in this data) to a
scale that's always available, since pixel-level replication within a well is
abundant even when well-level replication isn't:

1. **intra-well pixel** -- pairwise distance among pixels within the same well
   (measurement noise floor).
2. **inter-well, same chip, same target** -- distance between different wells' mean
   curves, when they share a chip and a target (only where >=2 such wells exist).
3. **inter-chip, same target** -- distance between different chips' mean curves for
   the same target -- the central "does the same target look different on different
   chips" number.
4. **inter-target, pooled** -- distance between different targets' overall means.

Ratios (3)/(2) and (3)/(1) answer "does chip add real structure beyond well/pixel
noise"; (4)/(3) answers "is target separation bigger than the chip effect within one
target" -- directly the thing a cross-chip classifier needs to be true.

In [ ]:
def _mean_pairwise_dist(mat):
    if len(mat) < 2:
        return np.nan
    return pdist(mat).mean()


def _groupby_idx(keys_iter):
    groups = {}
    for i, k in enumerate(keys_iter):
        groups.setdefault(k, []).append(i)
    return groups.items()


def compute_distance_summary(group_name, pool, n_sub=150, seed=0):
    curves, y_mapped, dataset_id, well_ids = pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["well_ids"]
    rng = np.random.default_rng(seed)

    intra_well = []
    if well_ids is not None:
        for (chip, w), idx in _groupby_idx(zip(dataset_id, well_ids)):
            idx_sub = _sub_idx(idx, n_sub, rng)
            d = _mean_pairwise_dist(curves[idx_sub])
            if not np.isnan(d):
                intra_well.append(d)
    lvl1 = np.nanmean(intra_well) if intra_well else np.nan

    inter_well_same_label = []
    if well_ids is not None:
        for (chip, lab), idx in _groupby_idx(zip(dataset_id, y_mapped)):
            idx = np.asarray(idx)
            wells_here = np.unique(well_ids[idx])
            if len(wells_here) < 2:
                continue
            well_means = np.stack([curves[idx[well_ids[idx] == w]].mean(axis=0) for w in wells_here])
            inter_well_same_label.append(_mean_pairwise_dist(well_means))
    lvl2 = np.nanmean(inter_well_same_label) if inter_well_same_label else np.nan

    inter_chip_same_label = []
    for lab in sorted(set(y_mapped)):
        idx = np.where(y_mapped == lab)[0]
        chips_here = np.unique(dataset_id[idx])
        if len(chips_here) < 2:
            continue
        chip_means = np.stack([curves[idx[dataset_id[idx] == c]].mean(axis=0) for c in chips_here])
        inter_chip_same_label.append(_mean_pairwise_dist(chip_means))
    lvl3 = np.nanmean(inter_chip_same_label) if inter_chip_same_label else np.nan

    labels = sorted(set(y_mapped))
    label_means = np.stack([curves[y_mapped == lab].mean(axis=0) for lab in labels])
    lvl4 = _mean_pairwise_dist(label_means) if len(labels) >= 2 else np.nan

    def _ratio(a, b):
        return a / b if (b is not None and not np.isnan(b) and b > 0) else np.nan

    return pd.DataFrame([{
        "group": group_name,
        "intra_well_pixel": lvl1,
        "inter_well_same_chip_same_label": lvl2,
        "inter_chip_same_label": lvl3,
        "inter_label_pooled": lvl4,
        "chip_vs_well_ratio": _ratio(lvl3, lvl2),
        "chip_vs_pixel_ratio": _ratio(lvl3, lvl1),
        "label_vs_chip_ratio": _ratio(lvl4, lvl3),
    }])


print("Distance-decomposition function defined.")

## 6. Quantitative -- raw-curve-space silhouette

Same idea as the earlier UMAP/t-SNE section's silhouette backup, but computed on a
PCA-reduced version of the actual curve space instead of a forced 2D projection --
avoids the question of whether a 2D layout distorted true separability. Three
scores: by well within each chip (intra-chip well separability, averaged across
chips), by chip within each target (the central question, averaged across targets),
by target pooled (overall separability, single number).

In [ ]:
def compute_silhouette_summary(group_name, pool, n_components=20, n_sub=300, seed=0):
    curves, y_mapped, dataset_id, well_ids = pool["curves"], pool["y_mapped"], pool["dataset_id"], pool["well_ids"]
    rng = np.random.default_rng(seed)

    n_comp = max(2, min(n_components, curves.shape[1], curves.shape[0] - 1))
    pca = PCA(n_components=n_comp, random_state=seed).fit(curves)
    evr = pca.explained_variance_ratio_.sum()

    def _sil(mask, group_arr):
        idx = np.where(mask)[0]
        idx_sub = _sub_idx(idx, n_sub, rng)
        labels_sub = group_arr[idx_sub]
        if len(set(labels_sub)) < 2:
            return np.nan
        X = pca.transform(curves[idx_sub])
        return silhouette_score(X, labels_sub)

    sil_by_well = []
    if well_ids is not None:
        for chip in sorted(set(dataset_id)):
            s = _sil(dataset_id == chip, well_ids)
            if not np.isnan(s):
                sil_by_well.append(s)
    sil_by_well_mean = np.nanmean(sil_by_well) if sil_by_well else np.nan

    sil_by_chip = []
    for lab in sorted(set(y_mapped)):
        s = _sil(y_mapped == lab, dataset_id)
        if not np.isnan(s):
            sil_by_chip.append(s)
    sil_by_chip_mean = np.nanmean(sil_by_chip) if sil_by_chip else np.nan

    sil_by_label = _sil(np.ones(len(y_mapped), dtype=bool), y_mapped)

    return pd.DataFrame([{
        "group": group_name, "pca_components": n_comp, "pca_explained_var": evr,
        "silhouette_by_well_within_chip": sil_by_well_mean,
        "silhouette_by_chip_within_label": sil_by_chip_mean,
        "silhouette_by_label_pooled": sil_by_label,
    }])


print("Silhouette function defined.")

## 7. Run -- loop over groups, plot + save + quantify

In [ ]:
distance_rows = []
silhouette_rows = []

for group_name in GROUPS_TO_TRY:
    print(f"\n{'='*70}\n{group_name}\n{'='*70}")
    pool = load_group_pool(group_name)
    if pool is None:
        print(f"  [SKIP] {group_name}: fewer than 2 usable datasets.")
        continue

    label_palette = build_label_palette(pool["y_mapped"])
    chip_palette = build_chip_palette(pool["dataset_id"])
    save_dir = Path(EXP_FOLDER) / "cross_dataset_cv" / group_name / SAVE_SUBDIR

    plot_intra_chip_by_well(group_name, pool, save_dir / f"{group_name}_1a_intra_chip_by_well.png")
    plot_intra_chip_by_label(group_name, pool, label_palette, save_dir / f"{group_name}_1b_intra_chip_by_label.png")
    plot_inter_chip_by_chip(group_name, pool, chip_palette, save_dir / f"{group_name}_2a_inter_chip_by_chip.png")
    plot_inter_chip_pooled(group_name, pool, label_palette, save_dir / f"{group_name}_2b_inter_chip_pooled.png")

    dist_df = compute_distance_summary(group_name, pool)
    sil_df = compute_silhouette_summary(group_name, pool)
    print(dist_df.to_string(index=False))
    print(sil_df.to_string(index=False))
    distance_rows.append(dist_df)
    silhouette_rows.append(sil_df)

distance_summary_df = pd.concat(distance_rows, ignore_index=True) if distance_rows else pd.DataFrame()
silhouette_summary_df = pd.concat(silhouette_rows, ignore_index=True) if silhouette_rows else pd.DataFrame()
print(f"\nDone: {len(distance_rows)} groups processed.")

## 8. Summary tables (all groups)

In [ ]:
distance_summary_df

In [ ]:
silhouette_summary_df